## **Imports**

In [ ]:
#!pip install -U transformers

## **Hugging Face Token**

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("my_hf_token")

In [2]:
from huggingface_hub import login
login(token=hf_token)

## **Imports**

In [3]:
import json
import re
import math
import random
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc, traceback

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
HF_TOKEN = hf_token

## **Load Model**

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,      # NOT torch_dtype (deprecated, as you already found)
    device_map="auto",
)
model.eval()

DEVICE = next(model.parameters()).device
NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded {MODEL_NAME}: {NUM_LAYERS} layers, hidden_dim={HIDDEN_DIM}, device={DEVICE}")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loaded meta-llama/Llama-3.2-3B-Instruct: 28 layers, hidden_dim=3072, device=cuda:0


In [6]:
# freeze model params

for p in model.parameters():
    p.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params in base model (should be 0): {trainable}")

Trainable params in base model (should be 0): 0


## **Dataset Loading**

In [7]:
df = pd.read_csv("/kaggle/input/datasets/anarvaaa/dataset-for-ssv/pico_dataset.csv")  # update path if running locally
# If running outside Kaggle, point this at your local copy instead, e.g.:
# df = pd.read_csv("pico_dataset.csv")
print(df.shape)
df.head(3)


(50, 7)


,srno,clinical_speciality,research_question,P,I,C,O
0,1,Cardiology,In adults with heart failure with preserved ej...,Adults with HFpEF (LVEF ≥ 50%),"SGLT2 inhibitors (e.g., empagliflozin, dapagli...",Placebo + standard care,Composite of cardiovascular death and hospital...
1,2,Cardiology,In patients undergoing percutaneous coronary i...,Adults undergoing PCI for acute coronary syndrome,Short-term DAPT (1–3 mo) followed by P2Y12 inh...,Standard DAPT for 12 months,Bleeding Academic Research Consortium (BARC) t...
2,3,Cardiology,In adults with severe asymptomatic aortic sten...,Adults with severe asymptomatic aortic stenosis,Early elective AVR (surgical or transcatheter),Conservative watchful waiting,All-cause mortality; myocardial infarction at ...


In [8]:
def format_target(row):
    return (
        f"P: {row['P']}\n"
        f"I: {row['I']}\n"
        f"C: {row['C']}\n"
        f"O: {row['O']}"
    )

def format_prompt(row):
    # Deliberately minimal / un-instructed prompt. We are NOT telling the model
    # to produce PICO format via prompting -- the steering vector is what should
    # push the model toward that structure.
    return f"{row['research_question'].strip()}\n"

examples = []
for _, row in df.iterrows():
    examples.append({
        "prompt": format_prompt(row),
        "target": format_target(row) + tokenizer.eos_token,
    })

random.shuffle(examples)
n_val = max(5, int(0.1 * len(examples)))
val_examples = examples[:n_val]
train_examples = examples[n_val:]
print(f"train={len(train_examples)}  val={len(val_examples)}")
print("--- sample ---")
print(train_examples[0]["prompt"])
print(train_examples[0]["target"])

train=45  val=5
--- sample ---
In adults aged 45 and older with degenerative medial meniscus tears and no severe osteoarthritis, does arthroscopic partial meniscectomy compared with structured physical therapy improve patient-reported pain and function at 12 months?

P: Adults aged ≥45 with symptomatic degenerative medial meniscus tears
I: Arthroscopic partial meniscectomy (APM)
C: Structured physical therapy/conservative exercise programs
O: Patient-reported knee pain and physical function (e.g., KOOS or WOMAC scores)<|eot_id|>


## **S-SPR**

In [30]:
class SPSR(nn.Module):
    """Single-layer Prompt Steering Replacement module (Eq. 7-8)."""
    def __init__(self, dim, dtype=torch.float32):
        super().__init__()
        self.z = nn.Parameter(torch.randn(dim, dtype=dtype) * 0.01)   # steering vector z_attr,l
        self.w = nn.Parameter(torch.randn(dim, dtype=dtype) * 0.01)   # probe weight w_attr,l
        self.b = nn.Parameter(torch.zeros(1, dtype=dtype) * 0.5)            # probe bias b_attr,l

    def coefficient(self, h):
        """lambda(h) = ReLU(h . w + b), computed per token. h: [..., dim] -> [..., 1]"""
        w = self.w.to(device=h.device, dtype=h.dtype)
        b = self.b.to(device=h.device, dtype=h.dtype)
        pre = torch.matmul(h, w) + b
        lam = F.leaky_relu(pre, negative_slope=0.01) if self.training else F.relu(pre)
        return lam.unsqueeze(-1)

    def forward(self, h, alpha):
        z = self.z.to(device=h.device, dtype=h.dtype)
        lam = self.coefficient(h)          # [..., 1], per-token scalar
        steered = h + alpha * lam * z
        return steered, lam

class SteeringHook:
    """Forward hook that injects an SPSR intervention into a decoder layer's output.

    `enabled=False` acts as a no-op passthrough, so the exact same model object
    can be used for steered vs. unsteered ("prompting-only") generation.
    Stores the last computed per-token lambda (`last_lambda`) so the training
    loop can apply the Sec. 3.5 regularization term.
    """
    def __init__(self, spsr: SPSR):
        self.sv = spsr
        self.enabled = False
        self.alpha = 0.0
        self.handle = None
        self.last_lambda = None

    def __call__(self, module, inputs, output):
        if not self.enabled:
            return output
        if isinstance(output, tuple):
            hidden_states = output[0]
            hidden_states, lam = self.sv(hidden_states, self.alpha)
            self.last_lambda = lam
            return (hidden_states,) + output[1:]
        else:
            hidden_states, lam = self.sv(output, self.alpha)
            self.last_lambda = lam
            return hidden_states

    def attach(self, layer_module):
        self.handle = layer_module.register_forward_hook(self)
        return self.handle

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

In [31]:
# Pick an intervention layer. Middle-ish layers tend to encode more abstract /
# format-level concepts (see Fig. 13 in the paper -- steerability varies by
# layer). Gemma-2B has NUM_LAYERS layers; we default to roughly the 40% mark.
# but for llama model, we implement layer 16 as recommended in table 4, appendix C
STEER_LAYER = 16
assert STEER_LAYER < NUM_LAYERS, f"Llama-3.2-3B-Instruct only has {NUM_LAYERS} layers"

# With device_map="auto" the model can be sharded across multiple GPUs, so the
# layer we hook may not live on the same device as model.parameters()[0]
# (which is what DEVICE points to). Find the *actual* device of the hooked
# layer and put the S-PSR module there -- the hook itself also casts on
# every call as a safety net, but starting on the right device avoids
# unnecessary cross-GPU copies.
layer_device = next(model.model.layers[STEER_LAYER].parameters()).device
print(f"Hooking layer index {STEER_LAYER} / {NUM_LAYERS} (on {layer_device})")

# NOTE: keep S-PSR params in float32 even though the model runs in float16.
# This is the standard "fp32 master weights" pattern for mixed precision:
# AdamW's default eps=1e-8 underflows to 0 once cast into fp16 (fp16's
# smallest normal value is ~6e-5), which silently produces NaNs via
# division-by-~0 in the Adam update. fp32 params + fp32 optimizer state
# avoids this; SPSR.forward already casts down to h.dtype only for the
# actual addition into the model's fp16 hidden states.
psr = SPSR(HIDDEN_DIM, dtype=torch.float32).to(layer_device)
steering_hook = SteeringHook(psr)
steering_hook.attach(model.model.layers[STEER_LAYER])

n_trainable = sum(p.numel() for p in psr.parameters() if p.requires_grad)
print(f"Trainable S-PSR params: {n_trainable}  (z, w: {HIDDEN_DIM} each, b: 1)")

Hooking layer index 16 / 28 (on cuda:1)
Trainable S-PSR params: 6145  (z, w: 3072 each, b: 1)


## **Batch Construction**

In [32]:
def build_batch(example, device=DEVICE):
    prompt_ids = tokenizer(example["prompt"], return_tensors="pt", add_special_tokens=True).input_ids[0]
    target_ids = tokenizer(example["target"], return_tensors="pt", add_special_tokens=False).input_ids[0]

    input_ids = torch.cat([prompt_ids, target_ids], dim=0)
    labels = input_ids.clone()
    labels[: len(prompt_ids)] = -100  # only train on the PICO-formatted continuation

    input_ids = input_ids.unsqueeze(0).to(device)
    labels = labels.unsqueeze(0).to(device)
    attention_mask = torch.ones_like(input_ids)
    return input_ids, attention_mask, labels

## **Training**

In [33]:
TRAIN_FACTORS = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0]
LR = 0.02
EPOCHS = 10
GRAD_CLIP = 1.0
LAMBDA_REG = 0.01   # weight of the Sec. 3.5 regularization term; set to 0.0 to disable

optimizer = torch.optim.AdamW(psr.parameters(), lr=LR, weight_decay=0.0)

def run_forward(input_ids, attention_mask, labels, alpha, lambda_reg=LAMBDA_REG):
    steering_hook.enabled = True
    steering_hook.alpha = alpha
    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = out.loss

    if lambda_reg > 0:
        lam = steering_hook.last_lambda  # [batch, seq, 1], computed on this forward pass
        resp_mask = (labels != -100).to(device=lam.device, dtype=lam.dtype).unsqueeze(-1)  # only response tokens
        lam_sum = (lam * resp_mask).sum()
        reg = torch.clamp(1.0 - lam_sum, min=0.0)
        loss = loss + lambda_reg * reg.to(loss.device)

    return loss

@torch.no_grad()
def evaluate(examples, alpha):
    steering_hook.enabled = True
    steering_hook.alpha = alpha
    losses = []
    for ex in examples:
        input_ids, attention_mask, labels = build_batch(ex)
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        losses.append(out.loss.item())
    return float(np.mean(losses))

In [34]:
history = {"train_loss": [], "val_loss": []}
skipped_steps = 0

for epoch in range(EPOCHS):
    psr.train()
    random.shuffle(train_examples)
    epoch_losses = []
    for ex in train_examples:
        input_ids, attention_mask, labels = build_batch(ex)
        alpha = random.choice(TRAIN_FACTORS)

        optimizer.zero_grad()
        loss = run_forward(input_ids, attention_mask, labels, alpha)

        if not torch.isfinite(loss):
            # Defensive: skip a bad step rather than let a NaN gradient poison
            # the Adam moment estimates for every subsequent step.
            skipped_steps += 1
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(psr.parameters(), GRAD_CLIP)
        optimizer.step()

        epoch_losses.append(loss.item())

    train_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
    # validate at the middle of the training factor range
    psr.eval()
    val_loss = evaluate(val_examples, alpha=TRAIN_FACTORS[len(TRAIN_FACTORS) // 2])
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"epoch {epoch+1:02d}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  (skipped {skipped_steps} bad steps so far)")

epoch 01/10  train_loss=1.5901  val_loss=1.3559  (skipped 0 bad steps so far)
epoch 02/10  train_loss=1.2131  val_loss=1.1997  (skipped 0 bad steps so far)
epoch 03/10  train_loss=1.1441  val_loss=1.3037  (skipped 0 bad steps so far)
epoch 04/10  train_loss=1.1640  val_loss=1.2334  (skipped 0 bad steps so far)
epoch 05/10  train_loss=1.2994  val_loss=1.4095  (skipped 0 bad steps so far)
epoch 06/10  train_loss=1.1104  val_loss=1.3379  (skipped 0 bad steps so far)
epoch 07/10  train_loss=1.2613  val_loss=1.2217  (skipped 0 bad steps so far)
epoch 08/10  train_loss=1.1949  val_loss=1.4237  (skipped 0 bad steps so far)
epoch 09/10  train_loss=1.3926  val_loss=1.5208  (skipped 0 bad steps so far)
epoch 10/10  train_loss=1.1008  val_loss=1.3230  (skipped 0 bad steps so far)


## **Best Steering Factor**

In [35]:
candidate_factors = [1.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 16.0, 20.0]
factor_scores = {}
for f in candidate_factors:
    factor_scores[f] = evaluate(val_examples, alpha=f)
    print(f"alpha={f:>5.1f}  val_loss={factor_scores[f]:.4f}")

BEST_ALPHA = min(factor_scores, key=factor_scores.get)
print(f"\nBest steering factor: {BEST_ALPHA}")

alpha=  1.0  val_loss=1.3112
alpha=  2.0  val_loss=1.2898
alpha=  4.0  val_loss=1.2879
alpha=  6.0  val_loss=1.3010
alpha=  8.0  val_loss=1.3152
alpha= 10.0  val_loss=1.3230
alpha= 12.0  val_loss=1.3274
alpha= 16.0  val_loss=1.3296
alpha= 20.0  val_loss=1.3306

Best steering factor: 4.0


In [36]:
@torch.no_grad()
def generate(prompt, alpha=None, max_new_tokens=80):
    steering_hook.enabled = alpha is not None
    steering_hook.alpha = alpha if alpha is not None else 0.0

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    steering_hook.enabled = False
    return text

In [37]:
for ex in val_examples[:3]:
    print("=" * 80)
    print("PROMPT:", ex["prompt"].strip())
    print("-" * 80)
    print("UNSTEERED (no hook, no PICO instruction):")
    print(generate(ex["prompt"], alpha=None))
    print("-" * 80)
    print(f"STEERED (S-PSR, alpha={BEST_ALPHA}):")
    print(generate(ex["prompt"], alpha=BEST_ALPHA))
    print("-" * 80)
    print("GROUND TRUTH:")
    print(ex["target"].replace(tokenizer.eos_token, ""))

PROMPT: In adults with out-of-hospital cardiac arrest (OHCA) and an initial shockable rhythm, does real-time audiovisual feedback during CPR compared with standard CPR without feedback improve survival to hospital discharge?
--------------------------------------------------------------------------------
UNSTEERED (no hook, no PICO instruction):
A study published in the New England Journal of Medicine found that real-time audiovisual feedback during CPR did not improve survival to hospital discharge compared to standard CPR without feedback.
The study, which was conducted by researchers at the University of California, San Francisco, and published in the New England Journal of Medicine, involved 1,000 adults who experienced OHCA and were treated with CPR. The participants were randomly
--------------------------------------------------------------------------------
STEERED (S-PSR, alpha=4.0):
P: Adults with OHCA and initial shockable rhythm
I: Real-time audiovisual feedback during CPR


## **Per token lambda**

In [38]:
@torch.no_grad()
def inspect_lambda(example, alpha):
    input_ids, attention_mask, labels = build_batch(example)
    steering_hook.enabled = True
    steering_hook.alpha = alpha
    _ = model(input_ids=input_ids, attention_mask=attention_mask)
    lam = steering_hook.last_lambda[0, :, 0].detach().float().cpu().numpy()
    steering_hook.enabled = False

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu().tolist())
    for tok, l in zip(tokens, lam):
        print(f"{tok!r:>20}  lambda={l:.4f}")

inspect_lambda(val_examples[0], alpha=BEST_ALPHA)

 '<|begin_of_text|>'  lambda=0.0000
                'In'  lambda=14.2188
           'Ġadults'  lambda=0.0000
             'Ġwith'  lambda=0.0000
              'Ġout'  lambda=0.0000
               '-of'  lambda=0.6670
                '-h'  lambda=0.0000
           'ospital'  lambda=0.0000
          'Ġcardiac'  lambda=0.0000
           'Ġarrest'  lambda=0.0000
                'Ġ('  lambda=0.0000
                'OH'  lambda=0.0000
                'CA'  lambda=0.0000
                 ')'  lambda=0.0000
              'Ġand'  lambda=0.0000
               'Ġan'  lambda=0.0000
          'Ġinitial'  lambda=0.0000
            'Ġshock'  lambda=0.0000
              'able'  lambda=0.0000
           'Ġrhythm'  lambda=0.0000
                 ','  lambda=0.0000
             'Ġdoes'  lambda=0.0000
             'Ġreal'  lambda=0.0000
             '-time'  lambda=0.0000
            'Ġaudio'  lambda=0.0000
            'visual'  lambda=0.0000
         'Ġfeedback'  lambda=0.0000
           'Ġduring'  lambd

In [39]:
torch.save(
    {
        "z": psr.z.detach().cpu(),
        "w": psr.w.detach().cpu(),
        "b": psr.b.detach().cpu(),
        "layer": STEER_LAYER,
        "best_alpha": BEST_ALPHA,
        "model_name": MODEL_NAME,
        "method": "S-PSR_LL",
    },
    "pico_spsr_vector.pt",
)
print("Saved pico_spsr_vector.pt")

Saved pico_spsr_vector.pt
